<div style="background-color:#d4edda;font-size:19px; color:#155724; padding:18px; border-radius:10px; border:1px solid #c3e6cb; font-family:Arial, sans-serif;">

<center> <b>Réalisé par A-ABO

<center> Ce projet a été réalisé par un étudiant en 2026.

<br>

<div style="font-size:9px; color:#4a5c4f; font-style:italic; text-align:center;">

<center> Ce projet peut contenir des erreurs.

</div>

</div>

<center> <h1> TD et TME 4: CREATION DE SCHEMAS-CONTRAINTES D'INTEGRITE </h1>


<center> <b> Installation H2 

<center>Le système utilisé pendant les TME est H2.

In [1]:
pip install psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


Télécharger le pilote de H2

**Relancez le kernel**: Kernel -> Restart kernel ...

In [2]:
import psycopg2
import os
local_dir = "./data"
os.makedirs(local_dir, exist_ok=True)
os.listdir(local_dir)

[]

# Démarrage du serveur H2
Démarrer un serveur de base de données H2 en arrière-plan, accessible uniquement localement(localhost, 127.0.0.1) sur le port 8082, avec les fichiers des bases de données stockés dans ./data (un fichier *.mv.db par base de données)

In [3]:
port = 8082
print(f'Le numero du port utilisé pour la connexion à la BD est: {port}')

Le numero du port utilisé pour la connexion à la BD est: 8082


In [4]:
%%bash --bg --out output -s "$port"
java -Dh2.bindAddress=127.0.0.1 -cp h2-2.1.214.jar:postgresql-42.6.0.jar org.h2.tools.Server -pg -pgPort $1 -baseDir ./data -ifNotExists

Couldn't find program: 'bash'


Tester si le serveur a été démarré

In [5]:
%%bash
nc -zv 127.0.0.1 8082

Couldn't find program: 'bash'


<span style="color:red"> 
<center> <h1>ADD THIS PART FOR WINDOWS !!

In [6]:
import psycopg2
import os
import socket
import subprocess
import time

In [7]:
local_dir = "./data"
os.makedirs(local_dir, exist_ok=True)
os.listdir(local_dir)

[]

In [8]:
port = 8082
print(f"Le numero du port utilisé pour la connexion à la BD est: {port}")

Le numero du port utilisé pour la connexion à la BD est: 8082


In [9]:
cmd = [
    "java",
    "-Dh2.bindAddress=127.0.0.1",
    "-cp",
    "h2-2.1.214.jar;postgresql-42.6.0.jar",
    "org.h2.tools.Server",
    "-pg",
    "-pgPort",
    str(port),
    "-baseDir",
    "./data",
    "-ifNotExists"
]

h2_process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

time.sleep(2)

if h2_process.poll() is None:
    print("H2 server started successfully in background.")
else:
    out, err = h2_process.communicate()
    print("H2 server failed to start.")
    print("STDOUT:\n", out)
    print("STDERR:\n", err)

H2 server started successfully in background.


In [10]:
s = socket.socket()
result = s.connect_ex(("127.0.0.1", port))
s.close()

if result == 0:
    print(f"Server is running on port {port}")
else:
    print(f"Server is NOT running on port {port}")

Server is running on port 8082


# Connexion du client au serveur H2
Se connecter au serveur H2 précédemment lancé en utilisant un nom de base de données, un login et un mot de passe. Si une connexion existante existe, elle est fermée avant d’en créer une nouvelle. Si la base n’existe pas encore, H2 la crée automatiquement dans le répertoire de données ./data (fichier *.mv.db). La fonction retourne un objet de connexion utilisable pour exécuter des commandes SQL sur le serveur.

In [11]:
def connect_H2(db,user,password):
    global connection
    try:
        connection
    except:
        connection = None
    if connection != None:
        try:
            connection.close()
            print("Connection closed")
        except  Error as e:
            print(f"The error '{e}' occurred")
    try:
        connection = psycopg2.connect(f"dbname={db} user={user} password={password} host=localhost port={port}")
        print("Connection to H2 DB successful")
    except Exception as e:
        print(f"The error '{e}' occurred")
    return connection

In [12]:
#Connexion à la base H2 nommée "jo<port>" avec l’utilisateur et le mot de passe spécifiés. Si la base n’existe pas encore, H2 la crée automatiquement. 
#Toute connexion précédente est fermée avant d’en établir une nouvelle.
connection = connect_H2("jo"+f'{port}',"user","pass")

Connection to H2 DB successful


# Fonction execute

Exécute une commande SQL sur la connexion fournie, affiche le nombre de lignes affectées et, si demandé, présente les résultats sous forme de tableau lisible avec les noms de colonnes. Le curseur peut être fermé automatiquement après l’exécution, et toute erreur est affichée. La fonction retourne le curseur utilisé pour accéder aux résultats ou None en cas d’échec.

In [13]:
def execute(connection, query, show=True, close=True, top=10):
    try:
        cursor = connection.cursor()
        cursor.execute(query)
        if cursor.rowcount >0:
          print(cursor.rowcount,"rows,", f"display only {top} rows")
        else:
          print(cursor.rowcount,"rows")
        if show and cursor.rowcount:
            names = [desc[0] for desc in cursor.description]

            lengths={}
            for attr in names:
                lengths[attr]=len(attr)
            for ligne in cursor:
                i=0
                for attr in ligne:
                    lengths[names[i]]=max(lengths[names[i]],len(str(attr)))
                    i=i+1
            print('|',end='')
            for attr in names:
                print(str(attr).ljust(lengths[attr]),end='|')
            print()
            print('|',end='')
            for attr in names:
                print(''.ljust(lengths[attr]+1,'-'),end='')
            print()
            cursor.execute(query)
            n=0
            for ligne in cursor:
                i=0
                print('|',end='')
                for attr in ligne:
                    print(str(attr)[:lengths[names[i]]].ljust(lengths[names[i]]),end='|')
                    i+=1
                print()
                n+=1
                if n>top :
                  break
        if close:
            cursor.close()
    except Exception as e:
        print(f"The error '{e}' occurred")
        cursor=None
    return cursor
print('fonction définie')

fonction définie


# Affichage du schéma d'une table

In [14]:
def show_table(connection,table_name):

    print(f'*********** Attributs de la table : {table_name} ************')
    query=f"""
        SELECT table_name,
              column_name,
              data_type,
              column_default,
              is_nullable,
              character_maximum_length,
              numeric_precision
        FROM information_schema.columns
        where lower(table_name)  = '{table_name}'
        """
    execute(connection,query)


    print(f'\n*********** Contraintes de la table : {table_name} ************')
    query = f"""
        SELECT tc.constraint_name, 
               tc.constraint_type, 
               cc.check_clause
        FROM information_schema.table_constraints tc
        LEFT JOIN information_schema.check_constraints cc 
               ON tc.constraint_name = cc.constraint_name
        WHERE lower(tc.table_name) = '{table_name}'
        AND tc.table_schema = 'public'
        ORDER BY tc.constraint_type
        """
    execute(connection, query)

# Affichage du schéma global 

In [15]:
def show_schema(connection):

    print('*********** Tables ************')
    query="""
    select TABLE_NAME
    from INFORMATION_SCHEMA.TABLES
    where TABLE_SCHEMA = 'public'
    """
    execute(connection,query,show=True,top=1000)

    print('\n\n*********** Domaines ************')
    query="""
    SELECT domain_name,check_clause
    FROM information_schema.domain_constraints a left outer join
         information_schema.check_constraints b
         on a.constraint_name=b.constraint_name
    """
    execute(connection,query,top=1000)

    print('\n\n*********** Attributs ************')
    query=f"""
    SELECT c.table_name,
           c.column_name,
           c.data_type,
           c.column_default,
           c.is_nullable,
           c.character_maximum_length,
           c.numeric_precision
    FROM INFORMATION_SCHEMA.TABLES t, information_schema.columns c
    where t.table_name=c.table_name
    and t.TABLE_SCHEMA = 'public'
    """
    execute(connection,query,top=1000)

    print('\n\n*********** Contraintes d''Intégrité ************')
    query="""
    SELECT 
        tc.table_name, 
        tc.constraint_name, 
        tc.constraint_type,
        cc.check_clause
    FROM information_schema.table_constraints tc
    LEFT JOIN information_schema.check_constraints cc 
        ON tc.constraint_name = cc.constraint_name
    WHERE tc.table_schema = 'public'
    ORDER BY tc.table_name, tc.constraint_type
    """
    execute(connection,query,top=1000)

# Exercices TD

On considère le schéma Entreprise décrit ci-dessous.

- EMPLOYE (<u>NumSS</u>, NomE, PrenomE, NumChef*, VilleE, DateNaiss) 
- PROJET(<u>NumProj</u>, NomProj, NomRespProj*, PrenomRespProj*, VilleP, Budget) 
- EMBAUCHE (<u>NumSS*, NumProj*</u>, DateEmb, Profil*) 
- GRILLE_SAL (<u>Profil</u>, salaire)


La clé primaire de chaque relation est soulignée et les attributs des clés étrangères sont suivis d’un astérisque. Cette base gère les employés et leurs affectations aux projets. Un employé est embauché dans un projet pour un profil donné (ex: 'Ingénieur', 'Analyste') et perçoit un salaire défini dans la grille salariale. Le chef d'un employé et le responsable d'un projet sont eux-mêmes des employés.  

**Contraintes d'unicité**
* PRIMARY KEY : Chaque table possède un identifiant unique non nul (souligné dans le schéma).
* UNIQUE : Il est interdit d'avoir deux employés ayant le même nom et le même prénom dans la table EMPLOYE. Pour permettre l'enregistrement d'employés dont l'identité est encore incomplète (ex: intérimaires dont seul le nom est connu), l'un ou l'autre de ces champs peut être laissé à NULL. En SQL, deux tuples ayant des NULL ne sont pas considérés comme des doublons.

**Contraintes d'intégrité référentielle (FOREIGN KEY)**
* Dans EMPLOYE, NumChef doit correspondre à un NumSS existant dans cette même table.
* Dans EMBAUCHE, NumSS référence la table EMPLOYE, NumProj référence la table PROJET et Profil référence la table GRILLE_SAL.
* Dans la table PROJET, le couple (NomRespProj, PrenomRespProj) est une clé étrangère faisant référence au couple (NomE, PrenomE) de la table EMPLOYE.


**Contraintes de domaine et de vérification (CHECK)**
* Types et Formats :
    * Le NumSS possède exactement 5 chiffres. Le NumProj varie entre 5 et 7 chiffres.
    * NomE, PrenomE, NomProj et Profil ne dépassent pas 20 caractères.
    * VilleE et VilleP ne dépassent pas 9 caractères et sont limitées à 'Paris', 'Lyon' ou 'Marseille'. 
    * Le Salaire peut avoir deux chiffres après la virgule et ne dépasse pas 90 000. Le Budget est un entier de 6 chiffres maximum (sans virgule)
* DEFAULT : La date d’embauche (DateEmb) peut être inconnue (valeur NULL). Si elle est omise lors de l’insertion, elle prend par défaut la date courante du système. La ville d’un employé peut être inconnue, sa valeur par défaut étant ‘Paris’.
* NOT NULL : Chaque projet doit obligatoirement avoir un nom et un responsable.
* Aucun employé ne peut avoir plus de 70 ans lors de son enregistrement (calculé entre DateNaiss et la date du jour).



## Création domaines
Créer les domaines définis en TD

In [27]:
#pour supprimer le schéma de la base courante
query="""
DROP ALL OBJECTS
"""
execute(connection,query,show=True)

# Le numéro de sécurité sociale possède exactement 5 chiffres
query="""
create domain nu_Sec as numeric(5)
"""
execute(connection,query,show=True)

# Les attributs textuels (NomE, PrenomE, NomProj, Profil) ne dépassent pas 20 caractères (ils peuvent en avoir moins)
query="""
create domain char_len as varchar(20)
"""
execute(connection,query,show=True)

# La ville d’un employé (VilleE) ou d’un projet (VilleP) se limite à 'Paris', 'Lyon' et 'Marseille' et sa longueur ne dépasse pas 9 caractères
query="""
create domain ch_Vil as varchar(9) check (value in ('Paris','Lyon' , 'Marseille'))
"""
execute(connection,query,show=True)

# Le numéro d’un projet varie entre 5 et 7 chiffres.
query="""
create domain nu_pro as integer
check (value between 10000 and 9999999)
"""
execute(connection,query,show=True)

0 rows
0 rows
0 rows
0 rows
0 rows


<cursor object at 0x0000016AAC76AC00; closed: -1>

<h5>Interroger le catalogue pour vérifier les domaines existants</h5>

In [28]:
query="""
select domain_name 
from INFORMATION_SCHEMA.DOMAINS 
"""
execute(connection,query,show=True)

4 rows, display only 10 rows
|domain_name|
|------------
|nu_pro     |
|char_len   |
|ch_vil     |
|nu_sec     |


<cursor object at 0x0000016AAC76AEA0; closed: -1>

## Création tables
Donner en SQL les instructions de création du schéma de la base en leur associant les contraintes de clés, référentielles et de domaines indiquées dans l'énoncé.

In [131]:
# Supprimer les tables s'elles existent
query="""
DROP TABLE IF EXISTS EMBAUCHE CASCADE;
DROP TABLE IF EXISTS PROJET CASCADE;
DROP TABLE IF EXISTS GRILLE_SAL CASCADE;
DROP TABLE IF EXISTS EMPLOYE CASCADE
"""
execute(connection,query,show=True)

# Grille_sal
query="""
create table Grille_sal(
    profil char_len,
    salaire numeric(7,2),
    primary key(profil)
)
"""
execute(connection,query,show=True)

# Employe
query="""
create table Employe(
    NumSS numeric(5),
    NomE char_len,
    PrenomE char_len,
    NumChef numeric(5),
    VilleE ch_vil,
    Datenaiss Date,
    primary key (NumSS),
    unique (NomE,PrenomE),
    foreign key (NumChef) references Employe,
    check (datediff( 'year',Datenaiss,current_date) <= 80)
)
"""
execute(connection,query,show=True)

# Projet
query="""
create table Projet(
    NumProj nu_pro,
    NomProj char_len,
    NomRespProj char_len,
    PrenomRespProj char_len,
    VilleP ch_vil,
    Budget numeric(6),
    primary key (NumProj),
     FOREIGN KEY (NomRespProj, PrenomRespProj)
        REFERENCES Employe(NomE, PrenomE)
    
)
"""
execute(connection,query,show=True)

# Embauche
query="""
create table Embauche(
    NumSS numeric(5),
    NumProj nu_pro,
    DateEmb DATE,
    Profil char_len,
    primary key (NumSS,NumProj),
    foreign key (NumSS) references Employe(NumSS),
    foreign key (NumProj) references Projet(NumProj),
    foreign key (Profil) references Grille_sal(Profil)
    )
"""
execute(connection,query,show=True)

0 rows
0 rows
0 rows
0 rows
0 rows


<cursor object at 0x0000016AACCB7220; closed: -1>

**Interroger le catalogue de données**

In [132]:
query="""
select TABLE_NAME
from INFORMATION_SCHEMA.TABLES 
where TABLE_SCHEMA = 'public'
"""
execute(connection,query,show=True)

4 rows, display only 10 rows
|table_name|
|-----------
|embauche  |
|employe   |
|projet    |
|grille_sal|


<cursor object at 0x0000016AACCB6F80; closed: -1>

**Afficher le schéma**

In [133]:
show_schema(connection)

*********** Tables ************
4 rows, display only 1000 rows
|table_name|
|-----------
|embauche  |
|employe   |
|projet    |
|grille_sal|


*********** Domaines ************
2 rows, display only 1000 rows
|domain_name|check_clause                          |
|---------------------------------------------------
|ch_vil     |VALUE IN('Paris', 'Lyon', 'Marseille')|
|nu_pro     |VALUE BETWEEN 10000 AND 9999999       |


*********** Attributs ************
18 rows, display only 1000 rows
|table_name|column_name   |data_type        |column_default|is_nullable|character_maximum_length|numeric_precision|
|------------------------------------------------------------------------------------------------------------------
|embauche  |numss         |numeric          |None          |NO         |None                    |5                |
|embauche  |numproj       |integer          |None          |NO         |None                    |32               |
|embauche  |dateemb       |date             |No

# Exercices TME

## Exercice 1
Reprendre le TD

In [134]:
query="""
insert into employe values('21456','LARS', 'Anna',null, 'Paris', parsedatetime('25-08-1975', 'dd-MM-yyyy' ))
"""
execute(connection,query,show=False)

1 rows, display only 10 rows


<cursor object at 0x0000016AACCB7AE0; closed: -1>

In [135]:
query="""
select * from employe
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
|numss|nome|prenome|numchef|villee|datenaiss |
|---------------------------------------------
|21456|LARS|Anna   |None   |Paris |1975-08-25|


<cursor object at 0x0000016AACCB7920; closed: -1>

In [136]:
query="""
delete from employe
"""
execute(connection,query,show=False)

1 rows, display only 10 rows


<cursor object at 0x0000016AACCB7E60; closed: -1>

## Exercice 2
Insérez dans chaque table au moins un n-uplet qui vérifie les contraintes d'intégrité. Vous avez la liberté de choisir les valeurs que vous voulez. Par exemple, l'instruction suivante insère un employé qui vérifie les contraintes

In [137]:
query="""
insert into employe(NUMSS,NOME,PRENOME, VILLEE , DATENAISS ) values ( '11111','Per1', 'Pper1', 'Lyon' , parsedatetime('21-02-2002','dd-MM-yyyy'))
"""
execute(connection,query,show=False)

1 rows, display only 10 rows


<cursor object at 0x0000016AACCB6180; closed: -1>

In [138]:
query="""
select * from employe
"""
execute(connection,query,show=True)

1 rows, display only 10 rows
|numss|nome|prenome|numchef|villee|datenaiss |
|---------------------------------------------
|11111|Per1|Pper1  |None   |Lyon  |2002-02-21|


<cursor object at 0x0000016AACCB7A00; closed: -1>

In [139]:
query="""
delete from employe
"""
execute(connection,query,show=False)

1 rows, display only 10 rows


<cursor object at 0x0000016AACCB7D80; closed: -1>

## Exercice 3:
Proposez des insertions qui violent les contraintes d'intégrité définies pour chaque table. Par exemple, l'instruction ci-dessous
viole la contrainte de clé primaire de Employe car elle tente d'insérer un employé sans valeur pour l'attribut clé primaire.

In [140]:
query="""
insert into employe values( null, 'Per2', 'Pper2',null , 'Paris',  parsedatetime('21-02-2002','dd-MM-yyyy') )

"""
execute(connection,query,show=False)

The error 'NULL not allowed for column "numss"; SQL statement:

insert into employe values( null, 'Per2', 'Pper2',null , 'Paris',  parsedatetime('21-02-2002','dd-MM-yyyy')  [23502-214]
DETAIL:  org.h2.jdbc.JdbcSQLIntegrityConstraintViolationException: NULL not allowed for column "numss"; SQL statement:

insert into employe values( null, 'Per2', 'Pper2',null , 'Paris',  parsedatetime('21-02-2002','dd-MM-yyyy')  [23502-214]
' occurred


### Proposer une insertion dans la table Employé qui ne respecte pas la contrainte de clé primaire.

In [141]:
query="""
insert into employe values( '1234567', 'Per2', 'Pper2',null , 'Paris',  parsedatetime('21-02-2002','dd-MM-yyyy') )

"""
execute(connection,query,show=True)

The error 'Value too long for column "numss NUMERIC(5)": "'1234567' (7)"; SQL statement:

insert into employe values( '1234567', 'Per2', 'Pper2',null , 'Paris',  parsedatetime('21-02-2002','dd-MM-yyyy')  [22001-214]
DETAIL:  org.h2.jdbc.JdbcSQLDataException: Value too long for column "numss NUMERIC(5)": "'1234567' (7)"; SQL statement:

insert into employe values( '1234567', 'Per2', 'Pper2',null , 'Paris',  parsedatetime('21-02-2002','dd-MM-yyyy')  [22001-214]
' occurred


### Proposer une insertion dans la table Employe qui ne respecte pas la contrainte de limite d'âge.

In [142]:
query="""
insert into employe values( '12345', 'Per2', 'Pper2',null , 'Paris',  parsedatetime('21-02-1700','dd-MM-yyyy') )

"""
execute(connection,query,show=True)

The error 'Check constraint violation: "CONSTRAINT_9F32: "; SQL statement:

insert into employe values( '12345', 'Per2', 'Pper2',null , 'Paris',  parsedatetime('21-02-1700','dd-MM-yyyy')  [23513-214]
DETAIL:  org.h2.jdbc.JdbcSQLIntegrityConstraintViolationException: Check constraint violation: "CONSTRAINT_9F32: "; SQL statement:

insert into employe values( '12345', 'Per2', 'Pper2',null , 'Paris',  parsedatetime('21-02-1700','dd-MM-yyyy')  [23513-214]
' occurred


### Proposer une insertion dans la table Employe qui ne respecte pas la contrainte de longueur de l'attribut NumSS.

In [143]:
query="""
insert into employe values('12345' , 'THIS_IS_TOOO_LONG_FOR_THE_NAME_MORE_THEN_20_CHAR','P',null,'Paris', parsedatetime('21-02-1700','dd-MM-yyyy'))
"""
execute(connection,query,show=True)

The error 'Value too long for column "nome CHARACTER VARYING(20)": "'THIS_IS_TOOO_LONG_FOR_THE_NAME_MORE_THEN_20_CHAR' (48)"; SQL statement:

insert into employe values('12345' , 'THIS_IS_TOOO_LONG_FOR_THE_NAME_MORE_THEN_20_CHAR','P',null,'Paris', parsedatetime('21-02-1700','dd-MM-yyyy') [22001-214]
DETAIL:  org.h2.jdbc.JdbcSQLDataException: Value too long for column "nome CHARACTER VARYING(20)": "'THIS_IS_TOOO_LONG_FOR_THE_NAME_MORE_THEN_20_CHAR' (48)"; SQL statement:

insert into employe values('12345' , 'THIS_IS_TOOO_LONG_FOR_THE_NAME_MORE_THEN_20_CHAR','P',null,'Paris', parsedatetime('21-02-1700','dd-MM-yyyy') [22001-214]
' occurred


### Proposer une insertion dans la table Employe qui ne respecte pas la contrainte sur les villes possibles.

In [144]:
query="""
insert into employe values('12345','P2','P22',null,'DEADLAND',parsedatetime('21-02-1700','dd-MM-yyyy') )
"""
execute(connection,query,show=True)

The error 'Check constraint violation: "(VALUE IN('Paris', 'Lyon', 'Marseille'))"; SQL statement:

insert into employe values('12345','P2','P22',null,'DEADLAND',parsedatetime('21-02-1700','dd-MM-yyyy')  [23513-214]
DETAIL:  org.h2.jdbc.JdbcSQLIntegrityConstraintViolationException: Check constraint violation: "(VALUE IN('Paris', 'Lyon', 'Marseille'))"; SQL statement:

insert into employe values('12345','P2','P22',null,'DEADLAND',parsedatetime('21-02-1700','dd-MM-yyyy')  [23513-214]
' occurred


### Insérer dans la table Employé deux employés avec le même nom et le même prénom.

In [145]:
query="""
insert into employe values('12347','P4','P44' , null , 'Paris', parsedatetime('21-02-1999','dd-MM-yyyy'));
insert into employe values('12344','P4','P44' , null , 'Paris', parsedatetime('21-02-1999','dd-MM-yyyy'))
"""
execute(connection,query,show=True)

The error 'Unique index or primary key violation: "public.CONSTRAINT_INDEX_9 ON public.employe(nome NULLS LAST, prenome NULLS LAST) VALUES ( /* key:3 */ 'P4', 'P44')"; SQL statement:

insert into employe values('12344','P4','P44' , null , 'Paris', parsedatetime('21-02-1999','dd-MM-yyyy') [23505-214]
DETAIL:  org.h2.jdbc.JdbcSQLIntegrityConstraintViolationException: Unique index or primary key violation: "public.CONSTRAINT_INDEX_9 ON public.employe(nome NULLS LAST, prenome NULLS LAST) VALUES ( /* key:3 */ 'P4', 'P44')"; SQL statement:

insert into employe values('12344','P4','P44' , null , 'Paris', parsedatetime('21-02-1999','dd-MM-yyyy') [23505-214]
' occurred


### Proposer une insertion dans la table Grille_SAL qui ne respecte pas la contrainte sur le salaire qui ne doit pas dépasser 90 000.

In [153]:
query="""

insert into Grille_sal values ('Pro', 100001)
"""
execute(connection,query,show=True)

The error 'Value too long for column "salaire NUMERIC(7, 2)": "100001 (32)"; SQL statement:


insert into Grille_sal values ('Pro', 10000 [22001-214]
DETAIL:  org.h2.jdbc.JdbcSQLDataException: Value too long for column "salaire NUMERIC(7, 2)": "100001 (32)"; SQL statement:


insert into Grille_sal values ('Pro', 10000 [22001-214]
' occurred


### Proposer une insertion dans la table Projet qui ne respecte pas la contrainte référentielle vers Employe : insérer un responsable de projet qui n'est pas dans la table Employé

In [156]:
query="""
insert into projet values('54321','NONAME','IM_NOT_IN_EMP','MEE_TOO','Paris','40000')
"""

execute(connection,query,show=True)

The error 'Referential integrity constraint violation: "CONSTRAINT_C59: public.projet FOREIGN KEY(nomrespproj, prenomrespproj) REFERENCES public.employe(nome, prenome) ('IM_NOT_IN_EMP', 'MEE_TOO')"; SQL statement:

insert into projet values('54321','NONAME','IM_NOT_IN_EMP','MEE_TOO','Paris','40000' [23506-214]
DETAIL:  org.h2.jdbc.JdbcSQLIntegrityConstraintViolationException: Referential integrity constraint violation: "CONSTRAINT_C59: public.projet FOREIGN KEY(nomrespproj, prenomrespproj) REFERENCES public.employe(nome, prenome) ('IM_NOT_IN_EMP', 'MEE_TOO')"; SQL statement:

insert into projet values('54321','NONAME','IM_NOT_IN_EMP','MEE_TOO','Paris','40000' [23506-214]
' occurred


### Proposer une insertion dans la table Embauche qui ne respecte pas une des contraintes référentielles : par exemple, associer un employé existant à un projet qui n'existe pas.

In [155]:
query="""
insert into embauche values ('12345','12348',parsedatetime('21-02-2004','dd-MM-yyyy'),'pro')
"""
execute(connection,query,show=True)

The error 'Referential integrity constraint violation: "CONSTRAINT_2E: public.embauche FOREIGN KEY(numss) REFERENCES public.employe(numss) (CAST(12345 AS NUMERIC(5)))"; SQL statement:

insert into embauche values ('12345','12348',parsedatetime('21-02-2004','dd-MM-yyyy'),'pro' [23506-214]
DETAIL:  org.h2.jdbc.JdbcSQLIntegrityConstraintViolationException: Referential integrity constraint violation: "CONSTRAINT_2E: public.embauche FOREIGN KEY(numss) REFERENCES public.employe(numss) (CAST(12345 AS NUMERIC(5)))"; SQL statement:

insert into embauche values ('12345','12348',parsedatetime('21-02-2004','dd-MM-yyyy'),'pro' [23506-214]
' occurred


## Exercice 4: Utilisation de ALTER TABLE

Le schéma de la base de données doit évoluer pour répondre à de nouvelles directives de gestion. Effectuez les modifications demandées en utilisant exclusivement la commande ALTER TABLE (ainsi que les commandes de gestion des contraintes et des tables si nécessaire pour résoudre les blocages). Chaque modification est indépendante et s’applique au schéma initial de la base. **Avant chaque question**, exécutez les cellules **Création domaines** et **Création tables** pour annuler les modifications des questions précédentes et garantir un environnement de test neutre.

Pour chacune des questions suivantes, vous devez :

* *Identifier le cas favorable (s'il existe)* : Insérez le minimum de données conformes pour illustrer une situation où la commande ALTER TABLE est acceptée par le SGBD.

* *Identifier le cas problématique (s'il existe)* : Insérez des données qui provoquent un rejet de la commande ALTER TABLE par le SGBD.
S'il existe une solution avec des instructions SQL uniquement au niveau schema (ALTER TABLE, DROP TABLE, sans utilisation de UPDATE ou DELETE) proposez et testez cette solution.


### Pour plus de clarté dans le schéma, on décide de renommer l'attribut VilleE en Ville_Residence dans la table EMPLOYE. Cette modification est-elle autorisée par le SGBD?

In [161]:
query="""
alter table Employe rename column VilleE to Ville_Residence
"""
execute(connection,query,show=False)

0 rows


<cursor object at 0x0000016AACCB6420; closed: -1>

<h4> Afficher le schéma de la table Employé</h4>

In [162]:
show_table(connection, 'employe')

*********** Attributs de la table : employe ************
6 rows, display only 10 rows
|table_name|column_name    |data_type        |column_default|is_nullable|character_maximum_length|numeric_precision|
|-------------------------------------------------------------------------------------------------------------------
|employe   |numss          |numeric          |None          |NO         |None                    |5                |
|employe   |nome           |character varying|None          |YES        |20                      |None             |
|employe   |prenome        |character varying|None          |YES        |20                      |None             |
|employe   |numchef        |numeric          |None          |YES        |None                    |5                |
|employe   |ville_residence|character varying|None          |YES        |9                       |None             |
|employe   |datenaiss      |date             |None          |YES        |None                  

### On décide que chaque projet doit impérativement être localisé. Il faut modifier l'attribut VilleP dans la table PROJET pour qu'il devienne NOT NULL. Que se passe-t-il si des projets existants n’ont pas de ville renseignée ?

<span style="color:green">

**10.4.2**  <b>
    
<b> Oui, seulement si aucune ligne existante n'a `VilleP = NULL`. Sinon, le SGBD refuse la modification.


### Pour faciliter la communication, ajoutez un attribut Email à la table EMPLOYE. La colonne est créée avec une valeur par défaut 'inconnu@entreprise.com'. La table contient déjà 100 employés. La modification est-elle autorisée ?

<span style="color:green">
    
**10.4.3**  <b>
    
 <b> Oui, la modification est autorisée. Les 100 employés existants recevront la valeur par défaut `inconnu@entreprise.com`.


### On décide que le budget d'un projet (Budget dans la table PROJET) ne peut jamais être inférieur à 10 000. Ajoutez cette contrainte. La modification est-elle autorisée si un projet en cours a un budget de 5 000 ?

<span style="color:green">

**10.4.4**  <b>
    
 <b> Non, la modification peut être refusée si un projet existant a un budget de `5000`, car il ne respecte pas `Budget >= 10000`.
</b>

### La direction décide de supprimer la limite d'âge de 70 ans pour les employés. Cette modification peut-elle être refusée par le SGBD ?

<span style="color:green">

**10.4.5**  <b>
    
 <b> Oui, la suppression de la contrainte est autorisée, car on enlève une restriction.



### La direction décide de ne plus stocker la date de naissance des employés. Il faut supprimer la colonne DateNaiss de la table EMPLOYE. Cette modification sera-t-elle acceptée ? 

<span style="color:green">

**10.4.6**  <b>
    
 <b> La modification peut être refusée si `DateNaiss` est utilisée dans une contrainte (ex: contrôle d’âge). Il faut d’abord supprimer cette contrainte.




### Pour économiser de l’espace, on souhaite réduire la taille de NomProj de 20 à 10 caractères. Que se passe-t-il si la table contient déjà un projet nommé 'Déploiement Fibre Optique' (25 caractères) ou même 'Projet Alpha' (12 caractères) ?

<span style="color:green">

**10.4.7** <b>
    
 <b> Non, la modification peut être refusée si des valeurs existantes dépassent 10 caractères, comme `'Déploiement Fibre Optique'` ou `'Projet Alpha'`.




### On veut supprimer le lien entre EMBAUCHE et GRILLE_SAL. La suppression est-elle autorisée ?

<span style="color:green">

**10.4.7** <b>
      
 <b> Oui, en principe c’est autorisé si on supprime la contrainte de clé étrangère. Mais il faut faire attention, car on perd alors le contrôle de cohérence entre `EMBAUCHE` et `GRILLE_SAL`.



### On veut supprimer la contrainte d'unicité sur (NomE, PrenomE) dans la table EMPLOYE. La suppression est-elle autorisée ?

<span style="color:green">

**10.4.9**  <b>
    
 <b> Non, la suppression peut être refusée si cette contrainte `UNIQUE (NomE, PrenomE)` est utilisée par une clé étrangère dans une autre table, comme `PROJET`.



### Dans la table PROJET, on décide de ne plus stocker le montant exact du budget, mais un code de tranche budgétaire (ex: 'PETIT', 'MOYEN', 'GROS'). Il faut donc changer le type de Budget(actuellement NUMERIC) en VARCHAR(10).

<span style="color:green">

**10.4.10**  
<b> La modification peut être refusée si la colonne `Budget` est utilisée dans une contrainte ou référencée par une autre table. Même si les anciennes valeurs numériques peuvent en général être converties en texte, il faut d’abord vérifier et éventuellement modifier/supprimer les dépendances (clés étrangères, contraintes, etc.).

</span>


### Supprimer la table Employe. La suppression est-elle autorisée ?

<span style="color:green">

**10.4.11**  

<b>Non, la suppression de la table `Employe` n’est pas autorisée directement si elle est référencée par des clés étrangères. Il faut d’abord supprimer les contraintes de clé étrangère, supprimer les tables dépendantes, ou utiliser `CASCADE`. [TD-9]

</span>

# Fermer la connexion

In [168]:
connection.commit() # implicit avec close
connection.close()

InterfaceError: connection already closed

In [167]:
connection

<connection object at 0x0000016AAC751140; dsn: 'user=user password=xxx dbname=jo8082 host=localhost port=8082', closed: 1>